## Long-Context Batch Extraction

With the latest gpt-4.1-mini and google gemini-flash


In [2]:
import os
import time
from collections import defaultdict

import pandas as pd
from litellm import completion, get_supported_openai_params
from pydantic import BaseModel

import agent_k.config.experiment_config as config_experiment
import agent_k.config.general as config_general
import agent_k.config.prompts_fast_n_slow as prompts_fast_n_slow
from agent_k.config.logger import logger
from agent_k.config.schemas import MineralSiteMetadataResponseAPI
from agent_k.notebooks.agentic_rag_v5 import count_tokens, create_markdown_retriever

In [3]:
def batch_extract_long_context(
    cdr_record_id: str,
    pydantic_model: BaseModel,
) -> str:
    """
    Extract entities from a PDF file by feeding the entire context to the model.
    """

    params = get_supported_openai_params(model=config_experiment.BATCH_EXTRACTION_MODEL)
    assert "response_format" in params, (
        "response_format is not supported for this model"
    )

    # Read with the parsed Markdown file
    with open(
        os.path.join("data/processed/43-101-refined", f"{cdr_record_id}.md"), "r"
    ) as f:
        context = f.read()

    input_tokens = count_tokens(context)
    logger.info(f"[Batch Extraction] Context has {input_tokens} tokens")

    logger.info("[Batch Extraction] Creating structured response")
    input = {
        "model": config_experiment.BATCH_EXTRACTION_MODEL,
        "response_format": pydantic_model,
        "messages": [
            {
                "role": "system",
                "content": prompts_fast_n_slow.PDF_AGENT_SYSTEM_PROMPT_STRUCTURED,
            },
            {
                "role": "user",
                "content": prompts_fast_n_slow.PDF_AGENT_USER_PROMPT_STRUCTURED.format(
                    context=context
                ),
            },
        ],
    }
    if config_experiment.BATCH_EXTRACTION_MODEL not in ["o3-mini", "o4-mini"]:
        input["temperature"] = config_experiment.BATCH_EXTRACTION_TEMPERATURE

    response = completion(**input)

    content = response.choices[0].message.content
    output_tokens = count_tokens(content)
    logger.info(f"[Batch Extraction] Response has {output_tokens} tokens")

    return content, input_tokens, output_tokens

## RAG-Based Extraction

Incorporate a retriever to compress the context to be more relevant.

In [4]:
def batch_extract_rag_based(
    cdr_record_id: str,
    pydantic_model: BaseModel,
) -> str:
    """
    Use a retriever to compress the context to be more relevant before feeding it to the model to extract structured information.
    """

    params = get_supported_openai_params(model=config_experiment.BATCH_EXTRACTION_MODEL)
    assert "response_format" in params, (
        "response_format is not supported for this model"
    )

    # Create a Vector Store from markdown file
    markdown_path = os.path.join("data/processed/43-101-refined", f"{cdr_record_id}.md")
    logger.info(f"[Batch Extraction] Creating Vector Store for {cdr_record_id}")
    retriever = create_markdown_retriever(
        markdown_path=markdown_path,
        collection_name=cdr_record_id,
        k=config_experiment.MAX_NUM_RETRIEVED_DOCS,
    )

    # Retrieve context for each fields in the pydantic model
    context = []
    for field_name, field_info in pydantic_model.model_fields.items():
        field_type = field_info.annotation
        default_value = field_info.default if field_info.default else "N/A"
        description = field_info.description
        question = prompts_fast_n_slow.QUESTION_TEMPLATE.format(
            field=field_name,
            dtype=field_type,
            default=default_value,
            description=description,
        )
        context.extend(retriever.invoke(question))
    logger.info(f"[Batch Extraction] Retrieved chunks in context: {len(context)}")

    input_tokens = count_tokens(str(context))
    logger.info(f"[Batch Extraction] Context has {input_tokens} tokens")

    logger.info("[Batch Extraction] Creating structured response")
    input = {
        "model": config_experiment.BATCH_EXTRACTION_MODEL,
        "response_format": pydantic_model,
        "messages": [
            {
                "role": "system",
                "content": prompts_fast_n_slow.PDF_AGENT_SYSTEM_PROMPT_STRUCTURED,
            },
            {
                "role": "user",
                "content": prompts_fast_n_slow.PDF_AGENT_USER_PROMPT_STRUCTURED.format(
                    context=context
                ),
            },
        ],
    }
    if config_experiment.BATCH_EXTRACTION_MODEL not in ["o3-mini", "o4-mini"]:
        input["temperature"] = config_experiment.BATCH_EXTRACTION_TEMPERATURE

    response = completion(**input)

    content = response.choices[0].message.content
    output_tokens = count_tokens(content)
    logger.info(f"[Batch Extraction] Response has {output_tokens} tokens")

    return content, input_tokens, output_tokens

In [5]:
batch_extract_func_map = {
    config_experiment.BatchExtractionMethod.LONG_CONTEXT: batch_extract_long_context,
    config_experiment.BatchExtractionMethod.RAG_BASED: batch_extract_rag_based,
}


def run_experiment(setup: config_experiment.BatchExtractionMethod):
    df_gt = pd.read_csv(config_general.INFERLINK_GROUND_TRUTH_TEST_VAL_PATH)

    rows = []
    tokens = defaultdict(int)
    start_time = time.time()
    for index, row in df_gt.iterrows():
        logger.info(f"Processing row {index + 1} of {len(df_gt)}")
        id, cdr_record_id = row["id"], row["cdr_record_id"]

        try:
            output, input_tokens, output_tokens = batch_extract_func_map[setup](
                cdr_record_id, MineralSiteMetadataResponseAPI
            )
            mineral_site_metadata = MineralSiteMetadataResponseAPI.model_validate_json(
                output
            )
            mineral_site_metadata = {
                "id": id,
                "cdr_record_id": cdr_record_id,
                **mineral_site_metadata.model_dump(),
            }
            rows.append(mineral_site_metadata)

            # Track tokens
            tokens["input_tokens"] += input_tokens
            tokens["output_tokens"] += output_tokens
        except Exception as e:
            logger.exception(
                f"Error processing row {index + 1} of {len(df_gt)} (Skipping): {e}"
            )

    end_time = time.time()

    df_pred = pd.DataFrame(rows)
    logger.info(
        f"Successfully extracted {len(df_pred)} reports. Write to experiment results directory."
    )
    # Generate timestamp once for both files
    output_dir = "data/experiments/250426_batch_extraction"

    # Save results to CSV
    timestamp = config_general.get_curr_ts()
    df_pred.to_csv(f"{output_dir}/{timestamp}_batch_extraction.csv", index=False)

    # Log experiment results
    logger.info(f"Experiment setup: {setup}")
    logger.info(f"Total time taken: {end_time - start_time} seconds")
    logger.info(f"Total input tokens: {tokens['input_tokens']}")
    logger.info(f"Total output tokens: {tokens['output_tokens']}")
    logger.info(
        f"Average input tokens per row: {tokens['input_tokens'] / len(df_gt):.2f}"
    )
    logger.info(
        f"Average output tokens per row: {tokens['output_tokens'] / len(df_gt):.2f}"
    )

    # Save experiment metadata to yaml file
    import yaml

    experiment_metadata = {
        "timestamp": timestamp,
        "setup": setup,
        "model": config_experiment.BATCH_EXTRACTION_MODEL,
        "temperature": config_experiment.BATCH_EXTRACTION_TEMPERATURE,
        "max_num_results": "N/A (long context)",
        "total_time_seconds": end_time - start_time,
        "total_input_tokens": tokens["input_tokens"],
        "total_output_tokens": tokens["output_tokens"],
        "average_input_tokens_per_row": tokens["input_tokens"] / len(df_gt),
        "average_output_tokens_per_row": tokens["output_tokens"] / len(df_gt),
        "num_rows_processed": len(df_pred),
        "total_rows": len(df_gt),
    }
    with open(f"{output_dir}/{timestamp}_experiment_metadata.yaml", "w") as f:
        yaml.dump(experiment_metadata, f)


if __name__ == "__main__":
    # ----------------------------------------------------------------------------------
    # Hyperparameters (configured manually in config/experiment_config.py)
    # ----------------------------------------------------------------------------------
    # Models: ["gpt-4o-mini", "gpt-4.1", "o4-mini"]
    # Methods: ["long_context", "rag_based"]
    # Temperature: [0.1]
    # Max retrieved results: [5] --> only applicable to rag_based
    run_experiment(config_experiment.BATCH_METHOD)

2025-04-27 17:27:17.922 | INFO     | __main__:run_experiment:17 - Processing row 1 of 50
2025-04-27 17:27:17.924 | INFO     | __main__:batch_extract_rag_based:14 - [Batch Extraction] Creating Vector Store for 0200a1c6d2cfafeb485d815d95966961d4c119e8662b8babec74e05b59ba4759d2
2025-04-27 17:27:17.959 | INFO     | agent_k.notebooks.self_rag_v5:create_markdown_retriever:103 - Number of tokens: 49301
2025-04-27 17:27:17.959 | INFO     | agent_k.notebooks.self_rag_v5:create_markdown_retriever:104 - Number of splits: 56
2025-04-27 17:27:17.960 | INFO     | agent_k.notebooks.self_rag_v5:create_markdown_retriever:105 - Average tokens per split: 880
2025-04-27 17:27:18.061 | INFO     | agent_k.notebooks.self_rag_v5:create_markdown_retriever:118 - Hashed collection name: rag-chroma_shard_5
2025-04-27 17:27:21.133 | INFO     | __main__:batch_extract_rag_based:34 - [Batch Extraction] Retrieved chunks in context: 35
2025-04-27 17:27:21.155 | INFO     | __main__:batch_extract_rag_based:37 - [Batch Ex